# Mask Classifier — 5-Model Comparison with ROC Curves (Colab, free GPU)

Trains and compares 5 CNN models (MobileNetV2, ResNet50, VGG16, InceptionV3, EfficientNetB0) on your face-mask dataset and reports accuracy, precision, recall, F1, **ROC-AUC**, inference speed and size — plus an **accuracy bar chart** and a combined **ROC-curve** figure.

**One-time setup:**
1. Upload your dataset zip (`archive__1_.zip`, containing `data/with_mask/` and `data/without_mask/`) to your Google Drive.
2. Menu: **Runtime -> Change runtime type -> T4 GPU -> Save**.

Then run each cell in order.

### 1. Check the GPU is on

In [ ]:
import tensorflow as tf
print('GPU available:', tf.config.list_physical_devices('GPU'))
!nvidia-smi -L

### 2. Connect Google Drive and unzip the dataset
Allow the pop-up. Assumes `archive__1_.zip` is in the top level of *My Drive*.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!unzip -q '/content/drive/MyDrive/archive__1_.zip' -d /content/dataset
import os
print('with_mask:', len(os.listdir('/content/dataset/data/with_mask')))
print('without_mask:', len(os.listdir('/content/dataset/data/without_mask')))

### 3. Train and evaluate all 5 models (~20-45 min on a T4)
Lower `EPOCHS` or shorten `MODELS` to go faster while testing.

In [ ]:
import tensorflow as tf, numpy as np, time, json, os
from tensorflow.keras import layers, Model
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, roc_curve, auc

DATA   = '/content/dataset/data'
MODELS = ['mobilenetv2', 'resnet50', 'vgg16', 'inceptionv3', 'efficientnetb0']
EPOCHS = 12
BATCH  = 32
OUT    = '/content/model_comparison'; os.makedirs(OUT, exist_ok=True)

K = tf.keras.applications
REGISTRY = {
    'mobilenetv2':    (K.MobileNetV2,    K.mobilenet_v2.preprocess_input, 224),
    'resnet50':       (K.ResNet50,       K.resnet.preprocess_input,       224),
    'vgg16':          (K.VGG16,          K.vgg16.preprocess_input,        224),
    'inceptionv3':    (K.InceptionV3,    K.inception_v3.preprocess_input, 299),
    'efficientnetb0': (K.EfficientNetB0, K.efficientnet.preprocess_input, 224),
}
# class_names order below sets label indices: with_mask = 0, without_mask = 1
def datasets(size):
    common = dict(validation_split=0.2, seed=42, image_size=(size, size),
                  batch_size=BATCH, label_mode='categorical',
                  class_names=['with_mask', 'without_mask'])
    tr = tf.keras.utils.image_dataset_from_directory(DATA, subset='training', **common)
    va = tf.keras.utils.image_dataset_from_directory(DATA, subset='validation', **common)
    return tr, va

results = {}
ROC = {}
for name in MODELS:
    print('\n' + '='*50 + f'\n  {name}\n' + '='*50)
    ctor, preprocess, size = REGISTRY[name]
    tr, va = datasets(size)
    AT = tf.data.AUTOTUNE
    trp = tr.map(lambda x, y: (preprocess(x), y)).prefetch(AT)
    vap = va.map(lambda x, y: (preprocess(x), y)).prefetch(AT)

    base = ctor(weights='imagenet', include_top=False, input_shape=(size, size, 3))
    base.trainable = False
    inp = layers.Input((size, size, 3))
    x = base(inp, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(2, activation='softmax')(x)
    model = Model(inp, out)
    model.compile(optimizer=Adam(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
    model.fit(trp, validation_data=vap, epochs=EPOCHS, verbose=2)

    # single-pass evaluation (labels, predictions and probabilities kept aligned)
    y_true, y_pred, scores = [], [], []
    for xb, yb in vap:
        pb = model.predict(xb, verbose=0)
        y_true.append(yb.numpy().argmax(1))
        y_pred.append(pb.argmax(1))
        scores.append(pb[:, 0])            # P(with_mask); with_mask is class index 0
    y_true = np.concatenate(y_true); y_pred = np.concatenate(y_pred); scores = np.concatenate(scores)
    print('  val class balance [with, without]:', np.bincount(y_true, minlength=2))

    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    acc = accuracy_score(y_true, y_pred)
    withmask = (y_true == 0).astype(int)   # 1 where truly with_mask
    fpr, tpr, _ = roc_curve(withmask, scores)
    roc_auc = auc(fpr, tpr)
    ROC[name] = (fpr, tpr, roc_auc)

    xb0 = next(iter(vap))[0]; t0 = time.time(); model.predict(xb0, verbose=0)
    ms = (time.time() - t0) / len(xb0) * 1000
    results[name] = dict(accuracy=float(acc), precision=float(p), recall=float(r), f1=float(f),
                         auc=float(roc_auc), ms_per_image=float(ms), params=int(model.count_params()))
    print(f'  {name}: acc={acc:.4f}  f1={f:.4f}  AUC={roc_auc:.4f}  {ms:.1f} ms/img')
    tf.keras.backend.clear_session()

json.dump(results, open(f'{OUT}/results.json', 'w'), indent=2)
print('\nDONE')

### 4. Comparison table, accuracy bar chart, and ROC curves

In [ ]:
import matplotlib.pyplot as plt
hdr = f"{'model':<16}{'acc':>8}{'prec':>8}{'recall':>8}{'f1':>8}{'AUC':>8}{'ms/img':>9}{'params(M)':>11}"
lines = [hdr, '-'*len(hdr)]
for n, m in sorted(results.items(), key=lambda kv: -kv[1]['auc']):
    lines.append(f"{n:<16}{m['accuracy']:>8.4f}{m['precision']:>8.4f}{m['recall']:>8.4f}"
                 f"{m['f1']:>8.4f}{m['auc']:>8.4f}{m['ms_per_image']:>9.1f}{m['params']/1e6:>11.1f}")
table = '\n'.join(lines)
open(f'{OUT}/comparison_table.txt', 'w').write(table)
print(table)

# accuracy bar chart
ns = list(results); accs = [results[n]['accuracy'] for n in ns]
plt.figure(figsize=(7, 4))
plt.bar(ns, accs, color='#1C6A8F')
plt.ylim(min(accs) - 0.05, 1.0); plt.ylabel('Test accuracy')
plt.title('Model comparison - accuracy'); plt.xticks(rotation=20)
for i, a in enumerate(accs): plt.text(i, a + 0.003, f'{a:.3f}', ha='center', fontsize=9)
plt.tight_layout(); plt.savefig(f'{OUT}/comparison_accuracy.png', dpi=150); plt.show()

# combined ROC curves
plt.figure(figsize=(6.2, 6))
for name, (fpr, tpr, a) in sorted(ROC.items(), key=lambda kv: -kv[1][2]):
    plt.plot(fpr, tpr, lw=2, label=f'{name} (AUC = {a:.3f})')
plt.plot([0, 1], [0, 1], '--', color='grey', lw=1, label='chance (AUC = 0.500)')
plt.xlim(0, 1); plt.ylim(0, 1.02)
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC curves - mask classifier model comparison')
plt.legend(loc='lower right'); plt.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f'{OUT}/roc_curves.png', dpi=150); plt.show()

### 5. Download your results (table, charts, ROC, JSON)

In [ ]:
from google.colab import files
for f in ['comparison_table.txt', 'comparison_accuracy.png', 'roc_curves.png', 'results.json']:
    files.download(f'{OUT}/{f}')